In [ ]:
# 1) 下载并解压 RealWaste（约 657 MB，需几分钟）
!wget -q --show-progress https://archive.ics.uci.edu/static/public/908/realwaste.zip -O realwaste.zip
!unzip -q -o realwaste.zip
# UCI 解压后通常为 realwaste-main/RealWaste/<ClassName>/*.jpg
from pathlib import Path
ROOT = Path('realwaste-main/RealWaste')
if not ROOT.exists():
    # 若结构不同，自动找 RealWaste 文件夹
    for p in Path('.').rglob('RealWaste'):
        if p.is_dir() and (p / 'Cardboard').exists():
            ROOT = p
            break
print('Data root:', ROOT.resolve(), 'exists:', ROOT.exists())

realwaste.zip           [              <=>   ] 656.65M  12.0MB/s    in 41s     
Data root: /content/realwaste-main/RealWaste exists: True


In [ ]:
# 2) 9 类文件夹名 -> 你的 3 类标签 0,1,2
REALWASTE_TO_3 = {
    'Cardboard': 0,
    'Glass': 0,
    'Metal': 0,
    'Paper': 0,
    'Plastic': 0,
    'Food Organics': 1,
    'Vegetation': 1,
    'Miscellaneous Trash': 2,
    'Textile Trash': 2,
}
TARGET_NAMES = ['Recyclables', 'Organics', 'Garbage']
SAMPLES_PER_CLASS = 50
SEED = 42

import random
from collections import defaultdict
random.seed(SEED)

pools = defaultdict(list)  # y in {0,1,2} -> list of file paths
for folder_name, y in REALWASTE_TO_3.items():
    d = ROOT / folder_name
    if not d.exists():
        print('Missing folder:', d)
        continue
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
        for f in d.glob(ext):
            pools[y].append(str(f))

for y in range(3):
    print(TARGET_NAMES[y], 'pool size:', len(pools[y]))

eval_paths = []  # (path, label)
for y in range(3):
    pool = pools[y]
    if len(pool) < SAMPLES_PER_CLASS:
        raise ValueError(f'Not enough images for class {y}: need {SAMPLES_PER_CLASS}, have {len(pool)}')
    chosen = random.sample(pool, SAMPLES_PER_CLASS)
    eval_paths.extend([(p, y) for p in chosen])

random.shuffle(eval_paths)
print('Total eval images:', len(eval_paths))

Recyclables pool size: 3092
Organics pool size: 847
Garbage pool size: 813
Total eval images: 150


In [ ]:
# 3) 可选：把抽样复制到文件夹，方便检查或写进报告插图
from shutil import copy2
OUT = Path('realwaste_eval_50x3')
for y, name in enumerate(TARGET_NAMES):
    (OUT / name).mkdir(parents=True, exist_ok=True)
idx = {0: 0, 1: 0, 2: 0}
for path, y in eval_paths:
    idx[y] += 1
    dst = OUT / TARGET_NAMES[y] / f'{idx[y]:03d}_{Path(path).name}'
    copy2(path, dst)
print('Copied to', OUT.resolve())

Copied to /content/realwaste_eval_50x3


In [ ]:
# 4) 与主项目相同的预处理 + WasteCNN（与 waste_classification_colab 一致）
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader

IMG_SIZE = 224
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class EvalPathsDataset(Dataset):
    def __init__(self, pairs, transform):
        self.pairs = pairs
        self.transform = transform
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        path, y = self.pairs[i]
        img = Image.open(path).convert('RGB')
        return self.transform(img), y

class WasteCNN(nn.Module):
    def __init__(self, num_classes=3, dropout=0.4):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.conv4 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2))
        self.conv5 = nn.Sequential(
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d(2))
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        x = self.conv5(self.conv4(self.conv3(self.conv2(self.conv1(x)))))
        x = self.avgpool(x)
        return self.fc(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True  # 固定 224 输入时加速卷积
print('device:', device)

model = WasteCNN(num_classes=3).to(device)

# 上传 cnn_best.pt 到 Colab 当前目录，或从 Drive 复制路径
CKPT = 'cnn_best.pt'  # 若保存的是 state_dict 纯权重：
state = torch.load(CKPT, map_location=device)
if isinstance(state, dict) and 'model' in state:
    model.load_state_dict(state['model'])
else:
    model.load_state_dict(state)
model.eval()
print('Loaded', CKPT)

device: cuda
Loaded cnn_best.pt


In [ ]:
# 5) 在 RealWaste 子集上评估
loader = DataLoader(EvalPathsDataset(eval_paths, val_tf), batch_size=32, shuffle=False, num_workers=2)
correct = 0
total = 0
per_class_correct = {0: 0, 1: 0, 2: 0}
per_class_total = {0: 0, 1: 0, 2: 0}

with torch.no_grad():
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.numel()
        for c in range(3):
            mask = y == c
            per_class_total[c] += mask.sum().item()
            per_class_correct[c] += ((pred == y) & mask).sum().item()

acc = correct / total
print(f'RealWaste subset accuracy (n={total}, 50 per class, seed={SEED}): {acc:.4f}')
for c in range(3):
    a = per_class_correct[c] / per_class_total[c] if per_class_total[c] else 0
    print(f"  {TARGET_NAMES[c]}: {a:.4f} ({per_class_correct[c]}/{per_class_total[c]})")

# 报告里可写：Kaggle test ~0.9515 vs RealWaste mapped subset ~acc（域偏移）

RealWaste subset accuracy (n=150, 50 per class, seed=42): 0.3467
  Recyclables: 0.9800 (49/50)
  Organics: 0.0600 (3/50)
  Garbage: 0.0000 (0/50)


In [ ]:
# 6a) 构建全量 (path, label)，并划分 train / val / test（70% / 15% / 15%）
import sys
import numpy as np
from collections import Counter

assert ROOT.exists(), '先运行下载解压那一格'

REALWASTE_TO_3 = {
    'Cardboard': 0, 'Glass': 0, 'Metal': 0, 'Paper': 0, 'Plastic': 0,
    'Food Organics': 1, 'Vegetation': 1,
    'Miscellaneous Trash': 2, 'Textile Trash': 2,
}
TARGET_NAMES = ['Recyclables', 'Organics', 'Garbage']

all_pairs = []
for folder_name, y in REALWASTE_TO_3.items():
    d = ROOT / folder_name
    if not d.exists():
        continue
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
        for f in d.glob(ext):
            all_pairs.append((str(f), y))

print('Total images:', len(all_pairs))
print('Class counts:', Counter(y for _, y in all_pairs))

rng = np.random.RandomState(42)
idx = rng.permutation(len(all_pairs))
n = len(idx)
n_test = int(0.15 * n)
n_val = int(0.15 * n)
n_train = n - n_test - n_val
tr_i, va_i, te_i = idx[:n_train], idx[n_train:n_train + n_val], idx[n_train + n_val:]

train_pairs = [all_pairs[i] for i in tr_i]
val_pairs = [all_pairs[i] for i in va_i]
test_pairs = [all_pairs[i] for i in te_i]
print(f'Split: train={len(train_pairs)}, val={len(val_pairs)}, test={len(test_pairs)}')

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.2),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = EvalPathsDataset(train_pairs, train_tf)
val_ds = EvalPathsDataset(val_pairs, val_tf)
test_ds = EvalPathsDataset(test_pairs, val_tf)

NUM_WORKERS = 0 if sys.platform == 'win32' else 2
_PIN = torch.cuda.is_available()
train_loader = DataLoader(
    train_ds, batch_size=32, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=_PIN)
val_loader = DataLoader(
    val_ds, batch_size=32, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=_PIN)
test_loader = DataLoader(
    test_ds, batch_size=32, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=_PIN)
print(f'DataLoaders: batch=32, num_workers={NUM_WORKERS}, pin_memory={_PIN}')

Total images: 4752
Class counts: Counter({0: 3092, 1: 847, 2: 813})
Split: train=3328, val=712, test=712
DataLoaders: batch=32, num_workers=2, pin_memory=True


In [ ]:
# 6b) 微调：从 cnn_best.pt 加载 + 类别权重 CrossEntropy + Adam
from torch.optim import Adam

model_ft = WasteCNN(num_classes=3).to(device)
state = torch.load(CKPT, map_location=device)
if isinstance(state, dict) and 'model' in state:
    model_ft.load_state_dict(state['model'])
else:
    model_ft.load_state_dict(state)

labels_train = [y for _, y in train_pairs]
cnt = Counter(labels_train)
n_tr = len(labels_train)
# 逆频率：n_tr / (3 * count_i)；某类在训练划分中缺失时用 1.0 占位
w_list = []
for i in range(3):
    c = cnt.get(i, 0)
    w_list.append(n_tr / (3 * c) if c > 0 else 1.0)
weights = torch.tensor(w_list, dtype=torch.float32, device=device)
print('Class weights:', weights.cpu().numpy(), 'counts:', dict(cnt))

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = Adam(model_ft.parameters(), lr=1e-4, weight_decay=1e-5)

def run_epoch(model, loader, train_mode):
    if train_mode:
        model.train()
    else:
        model.eval()
    tot_loss, correct, tot = 0.0, 0, 0
    ctx = torch.enable_grad() if train_mode else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train_mode:
                optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            if train_mode:
                loss.backward()
                optimizer.step()
            tot_loss += loss.item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            tot += y.size(0)
    return tot_loss / tot, correct / tot

EPOCHS = 15
best_val = 0.0
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model_ft, train_loader, True)
    va_loss, va_acc = run_epoch(model_ft, val_loader, False)
    print(f'Epoch {ep}/{EPOCHS} | train acc {tr_acc:.4f} | val acc {va_acc:.4f}')
    if va_acc > best_val:
        best_val = va_acc
        torch.save(model_ft.state_dict(), 'cnn_realwaste_finetuned.pt')
        print('  -> saved cnn_realwaste_finetuned.pt')

print('Best val acc:', best_val)

Class weights: [0.51050776 1.9093517  1.9326365 ] counts: {0: 2173, 2: 574, 1: 581}
Epoch 1/15 | train acc 0.7073 | val acc 0.7472
  -> saved cnn_realwaste_finetuned.pt
Epoch 2/15 | train acc 0.7251 | val acc 0.7949
  -> saved cnn_realwaste_finetuned.pt
Epoch 3/15 | train acc 0.6836 | val acc 0.7458
Epoch 4/15 | train acc 0.6526 | val acc 0.7219
Epoch 5/15 | train acc 0.6466 | val acc 0.7205
Epoch 6/15 | train acc 0.6632 | val acc 0.7121
Epoch 7/15 | train acc 0.6638 | val acc 0.7500
Epoch 8/15 | train acc 0.6863 | val acc 0.7317
Epoch 9/15 | train acc 0.6785 | val acc 0.7584
Epoch 10/15 | train acc 0.6902 | val acc 0.7416
Epoch 11/15 | train acc 0.7266 | val acc 0.7303
Epoch 12/15 | train acc 0.6899 | val acc 0.7163
Epoch 13/15 | train acc 0.7034 | val acc 0.7346
Epoch 14/15 | train acc 0.7073 | val acc 0.7640
Epoch 15/15 | train acc 0.7329 | val acc 0.7626
Best val acc: 0.7949438202247191


In [ ]:
model_ft.load_state_dict(torch.load('cnn_realwaste_finetuned.pt', map_location=device))
model_ft.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model_ft(x)
        y_true.extend(y.numpy().tolist())
        y_pred.extend(logits.argmax(1).cpu().numpy().tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
overall = (y_true == y_pred).mean()
print(f'Test accuracy (fine-tuned): {overall:.4f}')
for c in range(3):
    m = y_true == c
    if m.sum() > 0:
        acc_c = (y_pred[m] == c).mean()
        print(f'  {TARGET_NAMES[c]}: {acc_c:.4f} (n={m.sum()})')

Test accuracy (fine-tuned): 0.7556
  Recyclables: 0.9323 (n=458)
  Organics: 0.8030 (n=132)
  Garbage: 0.0410 (n=122)
